## Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from pathlib import Path

In [2]:
warnings.filterwarnings("ignore")

RAW       = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Standard schema for SPATIAL datasets (those with coords)
STANDARD_COLS = [
    "latitude", "longitude",
    "severity",        # int 1–4
    "killed",          # int
    "injured",         # int
    "weather",         # str
    "road_type",       # str
    "road_condition",  # str
    "lighting",        # str
    "speed_limit",     # float
    "state",           # str
    "country",         # str
    "year",            # int
    "month",           # str
    "hour",            # int
    "cause",           # str
    "vehicle_type",    # str
    "crash_type",      # str
    "area_type",       # str (urban/rural)
    "source",          # str
]


print(f"  Standard schema: {len(STANDARD_COLS)} columns")

  Standard schema: 20 columns


### Geocoding Lookups (Static - Zero api calls)

In [4]:
# All 27 named cities in Kaggle India + full 36 state capitals
CITY_COORDS = {
    "Ahmedabad":      (23.0225,  72.5714),
    "Bangalore":      (12.9716,  77.5946),
    "Chennai":        (13.0827,  80.2707),
    "Coimbatore":     (11.0168,  76.9558),
    "Durgapur":       (23.5204,  87.3119),
    "Dwarka":         (22.2394,  68.9678),
    "Jaipur":         (26.9124,  75.7873),
    "Jodhpur":        (26.2389,  73.0243),
    "Kanpur":         (26.4499,  80.3319),
    "Kolkata":        (22.5726,  88.3639),
    "Lucknow":        (26.8467,  80.9462),
    "Madurai":        (9.9252,   78.1198),
    "Mangalore":      (12.9141,  74.8560),
    "Mumbai":         (19.0760,  72.8777),
    "Mysore":         (12.2958,  76.6394),
    "Nagpur":         (21.1458,  79.0882),
    "New Delhi":      (28.6139,  77.2090),
    "Pune":           (18.5204,  73.8567),
    "Rohini":         (28.7041,  77.1025),
    "Siliguri":       (26.7271,  88.3953),
    "Surat":          (21.1702,  72.8311),
    "Tirupati":       (13.6288,  79.4192),
    "Udaipur":        (24.5854,  73.7125),
    "Vadodara":       (22.3072,  73.1812),
    "Varanasi":       (25.3176,  82.9739),
    "Vijayawada":     (16.5062,  80.6480),
    "Visakhapatnam":  (17.6868,  83.2185),
}

STATE_CAPITALS = {
    "Andhra Pradesh":       (15.9129,  79.7400),
    "Arunachal Pradesh":    (27.0844,  93.6053),
    "Assam":                (26.1445,  91.7362),
    "Bihar":                (25.5941,  85.1376),
    "Chandigarh":           (30.7333,  76.7794),
    "Chhattisgarh":         (21.2514,  81.6296),
    "Delhi":                (28.6139,  77.2090),
    "Goa":                  (15.2993,  74.1240),
    "Gujarat":              (23.0225,  72.5714),
    "Haryana":              (29.0588,  76.0856),
    "Himachal Pradesh":     (31.1048,  77.1734),
    "Jammu and Kashmir":    (34.0837,  74.7973),
    "Jharkhand":            (23.3441,  85.3096),
    "Karnataka":            (12.9716,  77.5946),
    "Kerala":               (8.5241,   76.9366),
    "Madhya Pradesh":       (23.2599,  77.4126),
    "Maharashtra":          (19.0760,  72.8777),
    "Manipur":              (24.6637,  93.9063),
    "Meghalaya":            (25.5788,  91.8933),
    "Mizoram":              (23.1645,  92.9376),
    "Nagaland":             (25.6751,  94.1086),
    "Odisha":               (20.2961,  85.8245),
    "Puducherry":           (11.9416,  79.8083),
    "Punjab":               (30.7333,  76.7794),
    "Rajasthan":            (26.9124,  75.7873),
    "Sikkim":               (27.3314,  88.6138),
    "Tamil Nadu":           (13.0827,  80.2707),
    "Telangana":            (17.3850,  78.4867),
    "Tripura":              (23.9408,  91.9882),
    "Uttar Pradesh":        (26.8467,  80.9462),
    "Uttarakhand":          (30.3165,  78.0322),
    "West Bengal":          (22.5726,  88.3639),
}


In [5]:

np.random.seed(42)

def geocode_row(city, state, jitter=0.25):
    """Return (lat, lng). City dict first, state capital fallback, jitter added."""
    city  = str(city).strip()
    state = str(state).strip()
    if city and city != "Unknown" and city in CITY_COORDS:
        lat, lng = CITY_COORDS[city]
    elif state and state in STATE_CAPITALS:
        lat, lng = STATE_CAPITALS[state]
    else:
        return None, None
    lat += np.random.uniform(-jitter, jitter)
    lng += np.random.uniform(-jitter, jitter)
    return round(lat, 6), round(lng, 6)

print(" Geocoding lookups ready")
print(f"   Named cities   : {len(CITY_COORDS)}")
print(f"   State capitals : {len(STATE_CAPITALS)}")

 Geocoding lookups ready
   Named cities   : 27
   State capitals : 32


## 2. Preprocess — India Mendeley (PRIMARY spatial data)

In [7]:
df_m = pd.read_csv(RAW / "india_mendeley" / "crashes.csv")
print(f"Raw Mendeley: {df_m.shape}  Columns: {list(df_m.columns)}")

Raw Mendeley: (2898, 17)  Columns: ['S. No.', 'Month ', 'Crash Date', 'Crash Day', 'Article Date ', 'Location', 'Million Plus City', 'State', 'LatLong', 'Vehicle 1', 'Vehicle/Object 2', 'Killed', 'Injured', 'Age', 'Gender', 'Road Type', 'Crash Type']


In [8]:
# ── Parse LatLong → separate lat/lng ─────────────────────────
coords = df_m["LatLong"].str.split(",", expand=True)
df_m["latitude"]  = pd.to_numeric(coords[0].str.strip(), errors="coerce")
df_m["longitude"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")

In [9]:
# ── Severity: Killed count → 1–4 scale ───────────────────────
def killed_to_severity(k):
    k = pd.to_numeric(k, errors="coerce")
    if pd.isna(k): return 2
    if k >= 3:     return 4  # mass fatal
    if k >= 1:     return 3  # fatal
    return 2                  # serious (injured but alive)

In [10]:
df_m["severity"] = df_m["Killed"].apply(killed_to_severity)

In [16]:
# ── Rename to standard schema ─────────────────────────────────
df_m = df_m.rename(columns={
    "State":      "state",
    "Killed":     "killed",
    "Injured":    "injured",
    "Road Type":  "road_type",
    "Crash Type": "crash_type",
    "Month ":      "month",
})

In [17]:
print(list(df_m.columns))

['S. No.', 'month', 'Crash Date', 'Crash Day', 'Article Date ', 'Location', 'Million Plus City', 'state', 'LatLong', 'Vehicle 1', 'Vehicle/Object 2', 'killed', 'injured', 'Age', 'Gender', 'road_type', 'crash_type', 'latitude', 'longitude', 'severity', 'country', 'source', 'weather', 'road_condition', 'lighting', 'speed_limit', 'year', 'hour', 'area_type', 'cause', 'vehicle_type']


In [18]:
# Fill standard cols not in Mendeley
df_m["country"]        = "India"
df_m["source"]         = "mendeley_india_2022_2023"
df_m["weather"]        = "unknown"
df_m["road_condition"] = "unknown"
df_m["lighting"]       = "unknown"
df_m["speed_limit"]    = np.nan
df_m["year"]           = 2022
df_m["hour"]           = -1
df_m["area_type"]      = "unknown"
df_m["cause"]          = df_m.get("crash_type", pd.Series(["unknown"]*len(df_m))).fillna("unknown")
df_m["vehicle_type"]   = df_m.get("Vehicle 1", pd.Series(["unknown"]*len(df_m))).fillna("unknown")

In [20]:
# ── Validate India bounds ─────────────────────────────────────
before = len(df_m)
df_m = df_m.dropna(subset=["latitude","longitude"])
df_m = df_m[df_m["latitude"].between(6, 38) &
            df_m["longitude"].between(65, 98)]

df_m["killed"]  = pd.to_numeric(df_m["killed"],  errors="coerce").fillna(0).astype(int)
df_m["injured"] = pd.to_numeric(df_m["injured"], errors="coerce").fillna(0).astype(int)

print(f"After coord validation: {len(df_m):,} / {before:,} rows kept")
df_mendeley_clean = df_m[STANDARD_COLS].copy()
print(f" Mendeley clean shape: {df_mendeley_clean.shape}")
df_mendeley_clean.head(3)

After coord validation: 2,898 / 2,898 rows kept
 Mendeley clean shape: (2898, 20)


,latitude,longitude,severity,killed,injured,weather,road_type,road_condition,lighting,speed_limit,state,country,year,month,hour,cause,vehicle_type,crash_type,area_type,source
0,18.278457,76.009574,4,4,0,unknown,NH,unknown,unknown,NaN,Maharashtra,India,2022,January,-1,Head On Collision,Car,Head On Collision,unknown,mendeley_india_2022_2023
1,24.543681,84.280571,4,6,18,unknown,NH,unknown,unknown,NaN,Jharkhand,India,2022,January,-1,Head On Collision,Pick Up,Head On Collision,unknown,mendeley_india_2022_2023
2,15.390068,73.854974,4,3,2,unknown,NH,unknown,unknown,NaN,Goa,India,2022,January,-1,Fixed Object Collision,Car,Fixed Object Collision,unknown,mendeley_india_2022_2023


### Preprocess - Kaggle India (geocoded)

In [21]:
kaggle_path = list((RAW / "kaggle_india").glob("*.csv"))[0]
df_k = pd.read_csv(kaggle_path)
print(f"Raw Kaggle India: {df_k.shape}  Columns: {list(df_k.columns)}")

Raw Kaggle India: (3000, 22)  Columns: ['State Name', 'City Name', 'Year', 'Month', 'Day of Week', 'Time of Day', 'Accident Severity', 'Number of Vehicles Involved', 'Vehicle Type Involved', 'Number of Casualties', 'Number of Fatalities', 'Weather Conditions', 'Road Type', 'Road Condition', 'Lighting Conditions', 'Traffic Control Presence', 'Speed Limit (km/h)', 'Driver Age', 'Driver Gender', 'Driver License Status', 'Alcohol Involvement', 'Accident Location Details']


In [22]:
# ── Geocode City Name + State Name → lat/lng ──────────────────
print("\nGeocoding rows (instant — no API calls)...")
lats, lngs = [], []
for _, row in df_k.iterrows():
    lat, lng = geocode_row(
        row.get("City Name", "Unknown"),
        row.get("State Name", "")
    )
    lats.append(lat)
    lngs.append(lng)

df_k["latitude"]  = lats
df_k["longitude"] = lngs

named_city = (df_k["City Name"] != "Unknown").sum()
print(f"  From city dict    : {named_city:,} rows")
print(f"  From state capital: {len(df_k)-named_city:,} rows")


Geocoding rows (instant — no API calls)...
  From city dict    : 862 rows
  From state capital: 2,138 rows


In [23]:
# ── Severity mapping ──────────────────────────────────────────
sev_map = {"Minor": 1, "Serious": 2, "Fatal": 3}
df_k["severity"] = df_k["Accident Severity"].map(sev_map).fillna(1).astype(int)

In [24]:
# ── Hour from "Time of Day" (HH:MM format) ───────────────────
df_k["hour"] = pd.to_datetime(
    df_k["Time of Day"], format="%H:%M", errors="coerce"
).dt.hour.fillna(-1).astype(int)

In [25]:
# ── Rename to standard schema ─────────────────────────────────
df_k = df_k.rename(columns={
    "State Name":                "state",
    "Month":                     "month",
    "Year":                      "year",
    "Weather Conditions":        "weather",
    "Road Type":                 "road_type",
    "Road Condition":            "road_condition",
    "Lighting Conditions":       "lighting",
    "Speed Limit (km/h)":        "speed_limit",
    "Number of Fatalities":      "killed",
    "Number of Casualties":      "injured",
    "Vehicle Type Involved":     "vehicle_type",
    "Accident Location Details": "crash_type",
    "Urban/Rural":               "area_type",
})

In [26]:
df_k["country"]    = "India"
df_k["source"]     = "kaggle_india_synthetic"
df_k["cause"]      = df_k.get("Accident Cause", pd.Series(["unknown"]*len(df_k))).fillna("unknown")
df_k["area_type"]  = df_k.get("area_type", pd.Series(["unknown"]*len(df_k))).fillna("unknown")
df_k["road_condition"] = df_k["road_condition"].fillna("unknown")
df_k["lighting"]   = df_k["lighting"].fillna("unknown")
df_k["speed_limit"]= pd.to_numeric(df_k.get("speed_limit"), errors="coerce")
df_k["killed"]     = pd.to_numeric(df_k.get("killed", pd.Series([0]*len(df_k))), errors="coerce").fillna(0).astype(int)
df_k["injured"]    = pd.to_numeric(df_k.get("injured", pd.Series([0]*len(df_k))), errors="coerce").fillna(0).astype(int)

In [27]:
before = len(df_k)
df_k = df_k.dropna(subset=["latitude","longitude"])
print(f"After geocoding : {len(df_k):,} / {before:,} rows have coords")

df_kaggle_clean = df_k[STANDARD_COLS].copy()
print(f" Kaggle India clean shape: {df_kaggle_clean.shape}")
df_kaggle_clean.head(3)

After geocoding : 3,000 / 3,000 rows have coords
 Kaggle India clean shape: (3000, 20)


,latitude,longitude,severity,killed,injured,weather,road_type,road_condition,lighting,speed_limit,state,country,year,month,hour,cause,vehicle_type,crash_type,area_type,source
0,34.020970,75.022657,2,4,0,Hazy,National Highway,Wet,Dark,61,Jammu and Kashmir,India,2021,May,1,unknown,Cycle,Curve,unknown,kaggle_india_synthetic
1,26.962697,80.995529,1,4,5,Hazy,Urban Road,Dry,Dusk,92,Uttar Pradesh,India,2018,January,21,unknown,Truck,Straight Road,unknown,kaggle_india_synthetic
2,21.079409,81.457597,1,5,6,Foggy,National Highway,Under Construction,Dawn,120,Chhattisgarh,India,2023,May,5,unknown,Pedestrian,Bridge,unknown,kaggle_india_synthetic


### Process Global Kaggle - Feature Stats Only
*No coordinates → extract feature pattern statistics for risk calibration.*
*This does NOT produce spatial rows. Output is a JSON lookup file.*

In [28]:
global_path = list((RAW / "global").glob("*.csv"))[0]
df_g = pd.read_csv(global_path, low_memory=False)
print(f"Raw Global: {df_g.shape}")
print(f"Columns: {list(df_g.columns)}")
print()
print(" CONFIRMED: No latitude/longitude columns in Global Kaggle.")
print(" Processing for feature pattern statistics only.")

Raw Global: (132000, 30)
Columns: ['Country', 'Year', 'Month', 'Day of Week', 'Time of Day', 'Urban/Rural', 'Road Type', 'Weather Conditions', 'Visibility Level', 'Number of Vehicles Involved', 'Speed Limit', 'Driver Age Group', 'Driver Gender', 'Driver Alcohol Level', 'Driver Fatigue', 'Vehicle Condition', 'Pedestrians Involved', 'Cyclists Involved', 'Accident Severity', 'Number of Injuries', 'Number of Fatalities', 'Emergency Response Time', 'Traffic Volume', 'Road Condition', 'Accident Cause', 'Insurance Claims', 'Medical Cost', 'Economic Loss', 'Region', 'Population Density']

 CONFIRMED: No latitude/longitude columns in Global Kaggle.
 Processing for feature pattern statistics only.


In [29]:
# Severity mapping for global
sev_map_g = {"Minor": 1, "Moderate": 2, "Severe": 3, "Fatal": 4}
df_g["severity_num"] = df_g["Accident Severity"].map(sev_map_g).fillna(2)

In [30]:
# ── Extract feature pattern stats ────────────────────────────
# These become lookup weights in feature engineering

# 1. Weather → avg severity
weather_risk = (df_g.groupby("Weather Conditions")["severity_num"]
                .agg(["mean","count"])
                .rename(columns={"mean":"avg_severity","count":"n"})
                .round(3))

# 2. Road type → avg severity
road_risk = (df_g.groupby("Road Type")["severity_num"]
             .agg(["mean","count"])
             .rename(columns={"mean":"avg_severity","count":"n"})
             .round(3))

In [31]:
# 3. Accident cause → avg severity
cause_risk = (df_g.groupby("Accident Cause")["severity_num"]
              .agg(["mean","count"])
              .rename(columns={"mean":"avg_severity","count":"n"})
              .round(3))

In [32]:
# 4. Speed limit bins → avg severity
df_g["speed_bin"] = pd.cut(df_g["Speed Limit"],
                            bins=[0,40,60,80,100,200],
                            labels=["0-40","41-60","61-80","81-100","100+"])
speed_risk = (df_g.groupby("speed_bin", observed=True)["severity_num"]
              .agg(["mean","count"])
              .rename(columns={"mean":"avg_severity","count":"n"})
              .round(3))

In [33]:
print("Weather → severity mapping:")
print(weather_risk.sort_values("avg_severity", ascending=False).to_string())
print()
print("Road type → severity mapping:")
print(road_risk.sort_values("avg_severity", ascending=False).to_string())
print()
print("Speed bins → severity mapping:")
print(speed_risk.to_string())

Weather → severity mapping:
                    avg_severity      n
Weather Conditions                     
Snowy                      2.004  26249
Foggy                      2.003  26137
Clear                      1.997  26426
Rainy                      1.996  26562
Windy                      1.995  26626

Road type → severity mapping:
           avg_severity      n
Road Type                     
Highway           1.999  43920
Main Road         1.999  44197
Street            1.999  43883

Speed bins → severity mapping:
           avg_severity      n
speed_bin                     
0-40              2.003  16099
41-60             1.994  29361
61-80             1.998  29277
81-100            2.001  29261
100+              2.001  28002


In [34]:
# Save as JSON lookup
feature_stats = {
    "weather_risk": weather_risk.reset_index().to_dict(orient="records"),
    "road_type_risk": road_risk.reset_index().to_dict(orient="records"),
    "cause_risk": cause_risk.reset_index().to_dict(orient="records"),
    "speed_risk": speed_risk.reset_index().rename(
        columns={"speed_bin":"bin"}).to_dict(orient="records"),
    "metadata": {
        "source": "global_kaggle",
        "total_rows": len(df_g),
        "purpose": "feature weight calibration — no spatial data"
    }
}

json_path = PROCESSED / "global_feature_stats.json"
with open(json_path, "w") as f:
    import json
    json.dump(feature_stats, f, indent=2)

print(f" Saved global_feature_stats.json")
print(f"   Path : {json_path}")
print(f"   Keys : {list(feature_stats.keys())}")
# print()
# print("These stats will be loaded in 03_feature_engineering.ipynb")
# print("to calibrate weather_risk, road_type_risk, and speed_risk scores.")

 Saved global_feature_stats.json
   Path : ..\data\processed\global_feature_stats.json
   Keys : ['weather_risk', 'road_type_risk', 'cause_risk', 'speed_risk', 'metadata']


### Build MoRTH Calibration Tables

In [35]:
morth_dir = RAW / "morth"

def clean_morth(df):
    for col in df.columns:
        if df[col].dtype == object:
            cleaned = df[col].astype(str).str.replace(",","").str.strip()
            numeric = pd.to_numeric(cleaned, errors="coerce")
            if numeric.notna().sum() > len(df) * 0.5:
                df[col] = numeric
    return df

In [36]:
df_sw_acc = clean_morth(pd.read_csv(morth_dir / "statewise_accidents_2019_2023.csv"))
df_sw_fat = clean_morth(pd.read_csv(morth_dir / "statewise_fatalities_2019_2023.csv"))

In [37]:
state_col_a = next(c for c in df_sw_acc.columns if "state" in c.lower() or "State" in c)
state_col_f = next(c for c in df_sw_fat.columns if "state" in c.lower() or "State" in c)

acc_2023 = next((c for c in df_sw_acc.columns if "2023" in str(c) and "Accident" in str(c)), None)
fat_2023 = next((c for c in df_sw_fat.columns if "2023" in str(c) and "Kill" in str(c)), None)

In [38]:
print(f"Accident 2023 col : {acc_2023}")
print(f"Fatality 2023 col : {fat_2023}")

Accident 2023 col : 2023 Accidents
Fatality 2023 col : 2023 Killed


In [39]:
morth = pd.DataFrame()
morth["state"] = df_sw_acc[state_col_a].astype(str).str.strip()

if acc_2023:
    morth["accidents_2023"] = pd.to_numeric(df_sw_acc[acc_2023], errors="coerce")
if fat_2023:
    fat_vals = pd.to_numeric(df_sw_fat[fat_2023].astype(str).str.replace(",",""), errors="coerce")
    morth["killed_2023"] = fat_vals.values[:len(morth)] if len(fat_vals) == len(morth) else np.nan

morth = morth.dropna(subset=["accidents_2023"])

# Normalise to 1.0–2.0 multiplier
morth["accident_rank"]    = morth["accidents_2023"].rank(pct=True)
morth["state_risk_weight"]= (1.0 + morth["accident_rank"]).round(3)

In [41]:
# Fatality rate
if "killed_2023" in morth.columns:
    morth["fatality_rate"] = (morth["killed_2023"] / morth["accidents_2023"]).round(4)

print()
print("State risk weights (top 10):")
print(morth[["state","accidents_2023","state_risk_weight"]]
      .sort_values("state_risk_weight", ascending=False).head(10).to_string(index=False))

morth.to_csv(PROCESSED / "state_risk_weights.csv", index=False)
print()
print(f" Saved state_risk_weights.csv ({len(morth)} states)")


State risk weights (top 10):
         state  accidents_2023  state_risk_weight
    Tamil Nadu         67213.0              2.000
Madhya Pradesh         55327.0              1.972
        Kerala         48091.0              1.944
 Uttar Pradesh         44534.0              1.917
     Karnataka         43440.0              1.889
   Maharashtra         35243.0              1.861
     Rajasthan         24694.0              1.833
     Telangana         22903.0              1.806
Andhra Pradesh         19949.0              1.778
       Gujarat         16349.0              1.750

 Saved state_risk_weights.csv (36 states)


### Merging Spatial Datasets

In [42]:
# Only Mendeley and Kaggle India go into the spatial merge
# Global Kaggle does NOT — it has no coordinates

In [44]:
frames = []
for name, df in [("Mendeley", df_mendeley_clean),
                  ("Kaggle India", df_kaggle_clean)]:
    print(f"  Adding {name}: {len(df):,} rows")
    frames.append(df)

merged = pd.concat(frames, ignore_index=True)
print(f"\nSpatial merged total: {len(merged):,} rows × {merged.shape[1]} cols")
print()
# print("Note: Global Kaggle (132K rows) NOT included here.")
# print("      Its feature stats are in global_feature_stats.json")
# print("      and will calibrate feature weights in notebook 03.")

  Adding Mendeley: 2,898 rows
  Adding Kaggle India: 3,000 rows

Spatial merged total: 5,898 rows × 20 cols



In [45]:
# Final standardisation
merged["severity"]      = merged["severity"].clip(1, 4).fillna(2).astype(int)
merged["killed"]        = merged["killed"].fillna(0).astype(int)
merged["injured"]       = merged["injured"].fillna(0).astype(int)
merged["hour"]          = merged["hour"].fillna(-1).astype(int)
merged["year"]          = merged["year"].fillna(2022).astype(int)
merged["weather"]       = merged["weather"].fillna("unknown").str.lower().str.strip()
merged["road_type"]     = merged["road_type"].fillna("unknown").str.lower().str.strip()
merged["road_condition"]= merged["road_condition"].fillna("unknown").str.lower().str.strip()
merged["lighting"]      = merged["lighting"].fillna("unknown").str.lower().str.strip()
merged["state"]         = merged["state"].fillna("unknown").str.strip()
merged["country"]       = merged["country"].fillna("India").str.strip()
merged["cause"]         = merged["cause"].fillna("unknown").str.lower().str.strip()
merged["vehicle_type"]  = merged["vehicle_type"].fillna("unknown").str.lower().str.strip()
merged["crash_type"]    = merged["crash_type"].fillna("unknown").str.lower().str.strip()
merged["area_type"]     = merged["area_type"].fillna("unknown").str.lower().str.strip()
merged["month"]         = merged["month"].fillna("unknown").str.strip()

In [46]:
# Drop invalid coords
before = len(merged)
merged = merged.dropna(subset=["latitude","longitude"])
merged = merged[merged["latitude"].between(-90,90) &
                merged["longitude"].between(-180,180)]
merged = merged.drop_duplicates(subset=["latitude","longitude","severity","source"])
print(f"After clean: {len(merged):,} / {before:,} rows kept")

After clean: 5,898 / 5,898 rows kept


In [47]:
# Composition summary
print("\n── Final spatial dataset composition ───────────────────")
comp = merged.groupby("source").agg(
    rows=("latitude","count"),
    avg_severity=("severity","mean"),
    total_killed=("killed","sum"),
).round(2)
print(comp.to_string())

print()
print("Severity distribution:")
sev_labels = {1:"Minor", 2:"Serious", 3:"Fatal", 4:"Mass Fatal"}
for k, label in sev_labels.items():
    n = (merged["severity"]==k).sum()
    pct = n / len(merged) * 100
    bar = "█" * int(pct / 2)
    print(f"  {k} {label:<12} {n:>6,} ({pct:5.1f}%) {bar}")


── Final spatial dataset composition ───────────────────
                          rows  avg_severity  total_killed
source                                                    
kaggle_india_synthetic    3000          1.98          7366
mendeley_india_2022_2023  2898          3.30          6584

Severity distribution:
  1 Minor         1,034 ( 17.5%) ████████
  2 Serious         981 ( 16.6%) ████████
  3 Fatal         3,013 ( 51.1%) █████████████████████████
  4 Mass Fatal      870 ( 14.8%) ███████


### Saving important outputs

In [48]:
# ── Save spatial parquet ──────────────────────────────────────
spatial_path = PROCESSED / "cleaned_spatial.parquet"
merged.to_parquet(spatial_path, index=False)
print(f" Saved cleaned_spatial.parquet")
print(f"   Shape : {merged.shape}")
print(f"   Size  : {spatial_path.stat().st_size / 1024:.1f} KB")

# ── Save preview CSV ──────────────────────────────────────────
merged.head(500).to_csv(PROCESSED / "cleaned_preview.csv", index=False)
print(" Saved cleaned_preview.csv (first 500 rows)")

print()
print("── All preprocessing outputs ────────────────────────────")
for f in sorted(PROCESSED.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size:>8.1f} KB")

 Saved cleaned_spatial.parquet
   Shape : (5898, 20)
   Size  : 158.4 KB
 Saved cleaned_preview.csv (first 500 rows)

── All preprocessing outputs ────────────────────────────
  cleaned_preview.csv                          85.3 KB
  cleaned_spatial.parquet                     158.4 KB
  eda_global_countries.png                     38.3 KB
  eda_global_features.png                      49.0 KB
  eda_global_severity.png                      19.8 KB
  eda_global_speed_severity.png                24.3 KB
  eda_kaggle_features.png                     125.2 KB
  eda_kaggle_hourly.png                        28.8 KB
  eda_mendeley_distributions.png               65.7 KB
  eda_mendeley_heatmap.html                   152.4 KB
  eda_mendeley_state_fatalities.png            85.7 KB
  eda_mendeley_vehicles.png                    37.5 KB
  eda_morth_top_states.png                     64.3 KB
  eda_severity_heatmap.png                     54.1 KB
  global_feature_stats.json                     1.9 KB

In [50]:
# Final validation
print("\n── Validation checks ────────────────────────────────────")
assert merged["latitude"].between(-90,90).all(),   "❌ Invalid latitudes!"
assert merged["longitude"].between(-180,180).all(),"❌ Invalid longitudes!"
assert merged["severity"].between(1,4).all(),      "❌ Severity out of range!"
assert merged["source"].notna().all(),             "❌ Source column has nulls!"
assert "latitude" not in df_g.columns,            "❌ Global should have no lat!"
assert len(merged) > 100,                         "❌ Too few spatial rows!"
print(" All validation checks passed")
print(f" {len(merged):,} spatial rows ready for Feature Engineering")
print(f" global_feature_stats.json ready for calibration")
print(f" state_risk_weights.csv ready for calibration")
# print("\nNext → 03_feature_engineering.ipynb")


── Validation checks ────────────────────────────────────
 All validation checks passed
 5,898 spatial rows ready for Feature Engineering
 global_feature_stats.json ready for calibration
 state_risk_weights.csv ready for calibration
